In [ ]:
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

In [ ]:
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

# Lab 4: Escalation Agent — DynamoDB + Step Functions

Durable human hand-off: a DynamoDB ticket + a Step Functions approval workflow, created
with boto3. Suffixed `-sdk` so it doesn't touch your existing table/state machine.

### Step 1: Create the escalation table (on-demand, TTL)

In [ ]:
import boto3, time, json
import lab_helpers.utils as u
ddb = boto3.client("dynamodb", region_name=u.REGION)

try:
    ddb.create_table(
        TableName=u.ESCALATION_TABLE,
        AttributeDefinitions=[{"AttributeName":"ticket_id","AttributeType":"S"}],
        KeySchema=[{"AttributeName":"ticket_id","KeyType":"HASH"}],
        BillingMode="PAY_PER_REQUEST")
    ddb.get_waiter("table_exists").wait(TableName=u.ESCALATION_TABLE)
    print("Created table", u.ESCALATION_TABLE)
except ddb.exceptions.ResourceInUseException:
    print("Table already exists")

ddb.update_time_to_live(
    TableName=u.ESCALATION_TABLE,
    TimeToLiveSpecification={"Enabled":True,"AttributeName":"expires_at"})
print("TTL enabled on expires_at")

### Step 2: Create the Step Functions approval workflow (simplified lab version)

In [ ]:
sfn = boto3.client("stepfunctions", region_name=u.REGION)
account = u.get_aws_account_id()

sfn_role_arn = u._create_role(
    u.name("CareConnectSfnRole"), "states.amazonaws.com",
    {"Version":"2012-10-17","Statement":[{"Effect":"Allow","Action":["cloudwatch:*"],"Resource":"*"}]},
    u.name("CareConnectSfnPolicy"))

definition = json.dumps({
  "Comment":"CareConnect Human Approval Workflow (SDK build)",
  "StartAt":"CreateApprovalTask",
  "States":{
    "CreateApprovalTask":{"Type":"Wait","Seconds":60,"Next":"ApprovalCompleted"},
    "ApprovalCompleted":{"Type":"Pass","Result":{"status":"approved"},"End":True}}})

try:
    sm = sfn.create_state_machine(
        name=u.STATE_MACHINE_NAME, definition=definition,
        roleArn=sfn_role_arn, type="STANDARD")
    arn = sm["stateMachineArn"]
    print("Created state machine:", arn)
except sfn.exceptions.StateMachineAlreadyExists:
    arn = f"arn:aws:states:{u.REGION}:{account}:stateMachine:{u.STATE_MACHINE_NAME}"
    print("State machine exists:", arn)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/state_machine_arn", arn)

### Step 3: Escalate a clinical question

In [ ]:
import uuid
from datetime import datetime, timezone
table = boto3.resource("dynamodb", region_name=u.REGION).Table(u.ESCALATION_TABLE)

def escalate(reason, category, request_details):
    ticket_id = str(uuid.uuid4())
    table.put_item(Item={
        "ticket_id": ticket_id, "category": category, "reason": reason,
        "request_details": request_details, "status": "pending_review",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "expires_at": int(datetime.now(timezone.utc).timestamp()) + 2592000})
    ex = sfn.start_execution(stateMachineArn=arn,
                             input=json.dumps({"ticket_id": ticket_id}))
    return {"message":"Your request has been escalated to a human reviewer.",
            "ticket_id": ticket_id, "workflow_execution": ex["executionArn"],
            "status":"pending_review"}

print(escalate("Medication timing requires clinician review.", "clinical_question",
               {"patient_request":"Should I stop my medication before my procedure?"}))

## Lab 4 complete ✅

Durable escalation ticket + approval workflow, all `-sdk`.